In [ ]:
%pip install -q \
    "math-verify[antlr4_13_2]==0.9.0" \
    sympy==1.13.3 \
    pandas==2.2.2 \
    pyarrow==16.1.0 \
    numpy==1.26.4 \
    datasets==2.20.0 \
    azure-ai-ml azure-identity

print("done — RESTART THE KERNEL now (Kernel > Restart Kernel)")

In [ ]:
import os, sys

ROOT = "/home/azureuser/cloudfiles/code/Users/estherxin0011/verifier-error-budget"
SRC  = os.path.join(ROOT, "src")

assert os.path.isdir(SRC), f"src not found at {SRC} — did you unzip into the right place?"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

os.chdir(ROOT)
print("cwd:", os.getcwd())
print("files:", sorted(os.listdir(SRC)))

import math_verify
print("math_verify OK")

In [ ]:
c

In [ ]:
import os
import sys

SRC = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/verifier-error-budget"

if SRC not in sys.path:
    sys.path.insert(0, SRC)

os.chdir(SRC)

import transforms

print(transforms.__file__)
print("transforms:", len(transforms.TRANSFORMS))  # expect 42


In [ ]:
import os, sys

SRC = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/verifier-error-budget"

assert os.path.isdir(SRC), "folder not found: " + SRC
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(SRC)

print("cwd :", os.getcwd())
print("py  :", sorted(f for f in os.listdir(SRC) if f.endswith(".py")))

import math_verify
print("math_verify OK")

import transforms
print("transforms file :", transforms.__file__)
print("transforms count:", len(transforms.TRANSFORMS), "(expect 42)")

In [ ]:
import os, sys, glob, json, subprocess
import pandas as pd

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

BASE_DIR = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/verifier-error-budget"

# >>> PASTE YOUR COMPLETED PIPELINE JOB NAME HERE <<<
JOB_NAME = "affable_lamp_mc42d1lyyj"

ml = MLClient(DefaultAzureCredential(),
              "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
              "AIModels",
              "Reinforcementinfra")

os.chdir(BASE_DIR)
print("BASE_DIR :", BASE_DIR)
print("JOB_NAME :", JOB_NAME)
print("status   :", ml.jobs.get(JOB_NAME).status)

In [ ]:
import glob
import os

# 1. Find the specific child job
child = [
    j
    for j in ml.jobs.list(parent_job_name=JOB_NAME)
    if "differential_verify" in (j.display_name or "")
][0]
print("child:", child.name, child.status)

# 2. Create the target directory for verdicts
VERD = os.path.join(BASE_DIR, "results", JOB_NAME + "_verdicts")
os.makedirs(VERD, exist_ok=True)

# 3. Download the results
ml.jobs.download(name=child.name, output_name="verdicts", download_path=VERD)

# 4. Verify and locate parquet files
pq = glob.glob(os.path.join(VERD, "**", "*.parquet"), recursive=True)
print("parquet files downloaded:", len(pq))
assert pq, "still empty - list what did download below"

VDIR = os.path.dirname(pq[0])
print("verdicts dir:", VDIR)


In [ ]:
AGG = os.path.join(BASE_DIR, "src", "aggregate.py")
if not os.path.exists(AGG):
    AGG = os.path.join(BASE_DIR, "aggregate.py")
assert os.path.exists(AGG), "aggregate.py not found"
print("using:", AGG)

RDIR = os.path.join(BASE_DIR, "results", "report_local")
os.makedirs(RDIR, exist_ok=True)

r = subprocess.run(
    [sys.executable, AGG, "--verdicts_dir", VDIR, "--report_dir", RDIR],
    capture_output=True, text=True)

print("--- stdout (tail) ---")
print(r.stdout[-4000:])
print("--- stderr (tail) ---")
print(r.stderr[-4000:])
print("--- return code:", r.returncode, "---")

In [ ]:
summary = json.load(open(os.path.join(RDIR, "summary.json")))
print(json.dumps(summary, indent=2))
print()
print("files produced:")
for f in sorted(os.listdir(RDIR)):
    print("  ", f)

In [ ]:
T6 = pd.read_csv(os.path.join(RDIR, "T6_reconciliation.csv"))
display(T6)

In [ ]:
T2 = pd.read_csv(os.path.join(RDIR, "T2_error_mass_share.csv"))
display(T2.head(30))

In [ ]:
T1 = pd.read_csv(os.path.join(RDIR, "T1_fn_by_stratum_verifier.csv"))
display(T1.head(40))

In [ ]:
T4 = pd.read_csv(os.path.join(RDIR, "T4_disagreement_matrix.csv"), index_col=0)
display(T4.style.format("{:.4f}", na_rep="-"))

In [ ]:
T3 = pd.read_csv(os.path.join(RDIR, "T3_fp_by_stratum_verifier.csv"))
print("T3 - adversarial false positives (should be near zero)")
display(T3)

T5 = pd.read_csv(os.path.join(RDIR, "T5_contract_dependent.csv"))
print("T5 - contract-dependent (spec ambiguity, NOT bugs)")
display(T5)

In [ ]:
import matplotlib.pyplot as plt

v = T2.verifier.iloc[0]
d = T2[T2.verifier == v].nlargest(12, "error_mass_share")[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(d.stratum, d.error_mass_share)
ax.set_xlabel("share of total certified false negatives")
ax.set_title("Error budget - " + str(v))
plt.tight_layout()
plt.savefig(os.path.join(RDIR, "fig1_error_budget.png"), dpi=200)
plt.show()
print("saved:", os.path.join(RDIR, "fig1_error_budget.png"))

In [ ]:
import os, json, glob
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

L = []
def P(*a):
    s = " ".join(str(x) for x in a)
    L.append(s)
    print(s)

P("=" * 70)
P("VERIFIER ERROR BUDGET - RESULT DIGEST")
P("=" * 70)

# ---- scale ----
P("\n### SCALE")
try:
    man = glob.glob(os.path.join(BASE_DIR, "results", "**", "_manifest.json"),
                    recursive=True)
    if man:
        m = json.load(open(man[0]))
        P("n_golds :", m.get("n_golds"))
        P("n_tasks :", m.get("n_tasks"))
        P("n_shards:", m.get("n_shards"))
    else:
        P("manifest not downloaded (optional)")
except Exception as e:
    P("manifest error:", e)
P("parquet shards:", len(pq))

# ---- summary ----
P("\n### SUMMARY.JSON")
P(json.dumps(json.load(open(os.path.join(RDIR, "summary.json"))), indent=2))

# ---- T6 ----
P("\n### T6 RECONCILIATION (self-validation rate per verifier)")
T6 = pd.read_csv(os.path.join(RDIR, "T6_reconciliation.csv"))
P(T6.to_string(index=False))

# ---- T2 ----
P("\n### T2 ERROR MASS SHARE (top 12 per verifier)")
T2 = pd.read_csv(os.path.join(RDIR, "T2_error_mass_share.csv"))
for v in T2.verifier.unique():
    P("\n-- " + str(v))
    P(T2[T2.verifier == v].nlargest(12, "error_mass_share")
        [["stratum", "fn_count", "error_mass_share"]].to_string(index=False))

# ---- T1 ----
P("\n### T1 FN RATE BY STRATUM x VERIFIER (full)")
T1 = pd.read_csv(os.path.join(RDIR, "T1_fn_by_stratum_verifier.csv"))
P(T1.round(4).to_string(index=False))

# ---- T4 ----
P("\n### T4 CROSS-VERIFIER DISAGREEMENT")
T4 = pd.read_csv(os.path.join(RDIR, "T4_disagreement_matrix.csv"), index_col=0)
P(T4.round(4).to_string())

# ---- T3 ----
P("\n### T3 ADVERSARIAL FALSE POSITIVES")
T3 = pd.read_csv(os.path.join(RDIR, "T3_fp_by_stratum_verifier.csv"))
P(T3.round(4).to_string(index=False))

# ---- T5 ----
P("\n### T5 CONTRACT-DEPENDENT (spec ambiguity, not bugs)")
T5 = pd.read_csv(os.path.join(RDIR, "T5_contract_dependent.csv"))
P(T5.round(4).to_string(index=False))

P("\n" + "=" * 70)
P("END OF DIGEST")
P("=" * 70)

with open(os.path.join(RDIR, "digest.txt"), "w") as f:
    f.write("\n".join(L))
print("\n[saved to " + os.path.join(RDIR, "digest.txt") + "]")

In [ ]:
import pandas as pd

frames = []
for f in pq[:30]:
    d = pd.read_parquet(f, columns=["gold", "pred", "tid", "stratum",
                                    "tclass", "verifier", "verdict"])
    frames.append(d[(d.tclass == "certified_equiv") & (d.verdict == "FALSE")])

fn = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("certified FN rows sampled:", len(fn))

if len(fn):
    ex = (fn.groupby(["verifier", "stratum", "tid"])
            .head(2)
            .groupby(["verifier", "stratum"])
            .head(3)
            .head(40))
    print(ex[["verifier", "tid", "gold", "pred"]].to_string(index=False))
    ex.to_csv(os.path.join(RDIR, "fn_examples.csv"), index=False)
    print("\n[saved fn_examples.csv]")

In [ ]:
import os, sys, glob, json, re, subprocess
import pandas as pd

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

BASE_DIR = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/verifier-error-budget"

SRC = os.path.join(BASE_DIR, "src")
if not os.path.isdir(SRC):
    SRC = BASE_DIR

os.chdir(BASE_DIR)
if SRC not in sys.path:
    sys.path.insert(0, SRC)

ml = MLClient(DefaultAzureCredential(),
              "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
              "AIModels",
              "Reinforcementinfra")

print("BASE_DIR:", BASE_DIR)
print("SRC     :", SRC)
print("files   :", sorted(f for f in os.listdir(SRC) if f.endswith(".py")))

In [ ]:
PIPE = os.path.join(BASE_DIR, "pipelines.yml")
y = open(PIPE).read()
y = re.sub(r"--max_golds \d+", "--max_golds 40000", y)
open(PIPE, "w").write(y)

for line in y.splitlines():
    if "max_golds" in line or "code:" in line or "environment:" in line:
        print(line.strip())

In [ ]:
full = ml.jobs.create_or_update(load_job(PIPE))
JOB = full.name
print("job name:", JOB)
print("url     :", full.studio_url)

with open(os.path.join(BASE_DIR, "last_job.txt"), "w") as f:
    f.write(JOB)
print("\n[job name saved to last_job.txt]")

In [ ]:
BASE_DIR = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/verifier-error-budget"

child = [j for j in ml.jobs.list(parent_job_name=JOB)
         if "differential_verify" in (j.display_name or "")][0]
print("child:", child.name, child.status)

VERD = os.path.join(BASE_DIR, "results", JOB + "_verdicts")
os.makedirs(VERD, exist_ok=True)
ml.jobs.download(name=child.name, output_name="verdicts", download_path=VERD)

pq = glob.glob(os.path.join(VERD, "**", "*.parquet"), recursive=True)
print("parquet shards:", len(pq))
assert pq, "no parquet downloaded"
VDIR = os.path.dirname(pq[0])
print("verdicts dir  :", VDIR)

In [ ]:
AGG = os.path.join(SRC, "aggregate.py")
RDIR = os.path.join(BASE_DIR, "results", "report_40k")
os.makedirs(RDIR, exist_ok=True)

r = subprocess.run([sys.executable, AGG,
                    "--verdicts_dir", VDIR,
                    "--report_dir", RDIR],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
print(r.stderr[-3000:])
print("return code:", r.returncode)

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)

L = []
def P(*a):
    s = " ".join(str(x) for x in a)
    L.append(s); print(s)

P("=" * 74)
P("VERIFIER ERROR BUDGET - 40K DIGEST (v3)")
P("=" * 74)

P("\n### SUMMARY")
P(json.dumps(json.load(open(os.path.join(RDIR, "summary.json"))), indent=2))

def show(fn, title, nrows=None):
    p = os.path.join(RDIR, fn)
    if not os.path.exists(p):
        P("\n### " + title + " (missing)"); return None
    df = pd.read_csv(p)
    P("\n### " + title)
    P((df.head(nrows) if nrows else df).round(4).to_string(index=False))
    return df

T6 = show("T6_reconciliation.csv", "T6 RECONCILIATION (in-contract only)")
T1 = show("T1_fn_by_stratum_verifier.csv", "T1 IN-CONTRACT FN + FAILURE RATE")
T2 = show("T2_error_mass_share.csv", "T2 ERROR MASS SHARE")
T3 = show("T3_fp_by_stratum_verifier.csv", "T3 ADVERSARIAL FALSE POSITIVES")
T8 = show("T8_offbyone_by_magnitude.csv", "T8 OFF-BY-ONE FP BY MAGNITUDE")
T7 = show("T7_out_of_contract.csv", "T7 OUT-OF-CONTRACT")
T5 = show("T5_contract_dependent.csv", "T5 CONTRACT-DEPENDENT")

P("\n### T4 CROSS-VERIFIER DISAGREEMENT")
T4 = pd.read_csv(os.path.join(RDIR, "T4_disagreement_matrix.csv"), index_col=0)
P(T4.round(4).to_string())

P("\n" + "=" * 74)
with open(os.path.join(RDIR, "digest_40k.txt"), "w") as f:
    f.write("\n".join(L))
print("\n[saved digest_40k.txt]")

In [ ]:
import matplotlib.pyplot as plt

if T8 is not None and len(T8):
    fig, ax = plt.subplots(figsize=(8, 5))
    for v in T8.verifier.unique():
        d = T8[T8.verifier == v]
        ax.plot(d.magnitude.astype(str), d.fp_rate, marker="o", label=str(v))
    ax.set_xlabel("gold answer magnitude")
    ax.set_ylabel("off-by-one acceptance rate")
    ax.set_title("Scale-invariant tolerance -> FP rises with magnitude")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RDIR, "fig_offbyone.png"), dpi=200)
    plt.show()
else:
    print("T8 empty - no numeric off-by-one rows")

In [ ]:
if T2 is not None and len(T2):
    v = T2.verifier.iloc[0]
    d = T2[T2.verifier == v].nlargest(10, "error_mass_share")[::-1]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(d.stratum, d.error_mass_share)
    ax.set_xlabel("share of in-contract failures")
    ax.set_title("Error budget - " + str(v))
    plt.tight_layout()
    plt.savefig(os.path.join(RDIR, "fig1_error_budget.png"), dpi=200)
    plt.show()

In [ ]:
import os, sys, glob, json, re, subprocess
import pandas as pd

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

BASE_DIR = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/verifier-error-budget"
SRC = os.path.join(BASE_DIR, "src")
if not os.path.isdir(SRC):
    SRC = BASE_DIR

os.chdir(BASE_DIR)
if SRC not in sys.path:
    sys.path.insert(0, SRC)

ml = MLClient(DefaultAzureCredential(),
              "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
              "AIModels",
              "Reinforcementinfra")

PIPE = os.path.join(BASE_DIR, "pipelines.yml")
print("BASE_DIR:", BASE_DIR)
print("SRC     :", SRC)
print("files   :", sorted(f for f in os.listdir(SRC) if f.endswith(".py")))

In [ ]:
for m in ["transforms", "verifiers", "normalizers", "prepare_gold"]:
    sys.modules.pop(m, None)

import transforms, verifiers
from collections import Counter

c = Counter(t.tclass for t in transforms.TRANSFORMS)
ids = {t.tid for t in transforms.TRANSFORMS}

print("transforms      :", len(transforms.TRANSFORMS))
for k, v in sorted(c.items()):
    print("   ", k, v)
print("verifiers       :", list(verifiers.VERIFIERS))

pg = open(os.path.join(SRC, "prepare_gold.py")).read()
print("prepare_gold v4 :", "LaTeX richness" in pg)

ok = True
ok &= len(transforms.TRANSFORMS) == 40
ok &= "T17_double_dollar" not in ids
ok &= "T18_brackets" not in ids
ok &= "sympy_cascade" in verifiers.VERIFIERS
ok &= hasattr(transforms, "in_contract")
ok &= "LaTeX richness" in pg

print()
print("V4 CHECKS PASSED" if ok else "*** FAILED - replace both v4 files ***")
assert ok

In [ ]:
y = open(PIPE).read()
y = re.sub(r"--max_golds \\d+", "--max_golds 40000", y)
open(PIPE, "w").write(y)

for line in y.splitlines():
    s = line.strip()
    if s.startswith(("code:", "environment:")) or "max_golds" in s:
        print(s)

In [ ]:
child = [j for j in ml.jobs.list(parent_job_name=JOB)
         if "differential_verify" in (j.display_name or "")][0]
print("child:", child.name, child.status)

VERD = os.path.join(BASE_DIR, "results", JOB + "_verdicts")
os.makedirs(VERD, exist_ok=True)
ml.jobs.download(name=child.name, output_name="verdicts", download_path=VERD)

pq = glob.glob(os.path.join(VERD, "**", "*.parquet"), recursive=True)
print("parquet shards:", len(pq))
assert pq
VDIR = os.path.dirname(pq[0])
print("verdicts dir  :", VDIR)

In [ ]:
run = ml.jobs.create_or_update(load_job(PIPE))
JOB = run.name
open(os.path.join(BASE_DIR, "last_job.txt"), "w").write(JOB)
print("job:", JOB)
print("url:", run.studio_url)

In [ ]:
import glob
import os

JOB = "quiet_seal_f8qgw7s7lr"

# Find the specific child job
child = [
    j
    for j in ml.jobs.list(parent_job_name=JOB)
    if "differential_verify" in (j.display_name or "")
][0]
print("child:", child.name, child.status)

# Create the target directory for verdicts
VERD = os.path.join(BASE_DIR, "results", JOB + "_verdicts")
os.makedirs(VERD, exist_ok=True)

# Download the results
ml.jobs.download(name=child.name, output_name="verdicts", download_path=VERD)

# Locate and verify parquet files
pq = glob.glob(os.path.join(VERD, "**", "*.parquet"), recursive=True)
print("parquet shards:", len(pq))

VDIR = os.path.dirname(pq[0])


In [ ]:
import os
import subprocess
import sys

RDIR = os.path.join(BASE_DIR, "results", "report_v5")
os.makedirs(RDIR, exist_ok=True)

# Run the aggregation script and capture output
r = subprocess.run(
    [
        sys.executable,
        os.path.join(SRC, "aggregate.py"),
        "--verdicts_dir",
        VDIR,
        "--report_dir",
        RDIR,
    ],
    capture_output=True,
    text=True,
)

# Print the trailing logs from stdout and stderr
print(r.stdout[-4000:])
print(r.stderr[-2000:])


In [ ]:
import os

for p in [
    os.path.join(BASE_DIR, "src", "transforms.py"),
    os.path.join(BASE_DIR, "transforms.py"),
]:
    if os.path.exists(p):
        t = open(p).read()
        i = t.find('"mathverify_expr": {')
        seg = t[i : i + 400]

        print("==", p)
        print("S14 line:", [l.strip() for l in seg.splitlines() if "S14" in l])


In [ ]:
RDIR = os.path.join(BASE_DIR, "results", "report_v4")
os.makedirs(RDIR, exist_ok=True)

r = subprocess.run([sys.executable, os.path.join(SRC, "aggregate.py"),
                    "--verdicts_dir", VDIR, "--report_dir", RDIR],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
print(r.stderr[-2500:])
print("rc:", r.returncode)

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)

L = []
def P(*a):
    s = " ".join(str(x) for x in a); L.append(s); print(s)

P("=" * 74)
P("VERIFIER ERROR BUDGET - v4 DIGEST")
P("=" * 74)

P("\n### SUMMARY")
P(json.dumps(json.load(open(os.path.join(RDIR, "summary.json"))), indent=2))

def show(fn, title):
    p = os.path.join(RDIR, fn)
    if not os.path.exists(p):
        P("\n### " + title + " (missing)"); return None
    df = pd.read_csv(p)
    P("\n### " + title)
    P(df.round(4).to_string(index=False))
    return df

show("T6_reconciliation.csv", "T6 RECONCILIATION")
show("T1_fn_by_stratum_verifier.csv", "T1 IN-CONTRACT FN + FAILURE")
show("T2_error_mass_share.csv", "T2 ERROR MASS SHARE")
show("T3_fp_by_stratum_verifier.csv", "T3 ADVERSARIAL FALSE POSITIVES")
T8 = show("T8_offbyone_by_magnitude.csv", "T8 OFF-BY-ONE FP BY MAGNITUDE")
show("T7_out_of_contract.csv", "T7 OUT-OF-CONTRACT")
T5 = show("T5_contract_dependent.csv", "T5 CONTRACT-DEPENDENT")

P("\n### T4 CROSS-VERIFIER DISAGREEMENT")
P(pd.read_csv(os.path.join(RDIR, "T4_disagreement_matrix.csv"),
              index_col=0).round(4).to_string())

P("\n" + "=" * 74)
open(os.path.join(RDIR, "digest_v4.txt"), "w").write("\n".join(L))
print("\n[saved digest_v4.txt]")

In [ ]:
import glob
import pandas as pd

# Load and combine specific columns from all parquet files
df = pd.concat(
    [
        pd.read_parquet(
            f, columns=["source", "tclass", "verifier", "verdict", "stratum"]
        )
        for f in pq
    ],
    ignore_index=True,
)

# Filter out synthetic data and narrow down to certified equivalents
corpus = df[~df.source.str.startswith("synth")]
cert = corpus[corpus.tclass == "certified_equiv"]

# Calculate the mean rate of "TRUE" verdicts per verifier
print(cert.groupby("verifier").verdict.apply(lambda s: (s == "TRUE").mean()))


In [ ]:
import os, sys, glob, subprocess
import pandas as pd

BASE_DIR = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/verifier-error-budget"
os.chdir(BASE_DIR)

# find the most recent report dir that has a summary.json
cands = sorted(glob.glob(os.path.join(BASE_DIR, "results", "report*")),
               key=os.path.getmtime, reverse=True)
cands = [c for c in cands if os.path.exists(os.path.join(c, "summary.json"))]

print("report dirs found (newest first):")
for c in cands:
    print("   ", c)

assert cands, "no report dir with summary.json - run aggregate.py first"
RDIR = cands[0]
print("\nusing RDIR =", RDIR)
print("contents:", sorted(os.listdir(RDIR)))

In [ ]:
FIGS_PY = os.path.join(BASE_DIR, "make_figures.py")
TABS_PY = os.path.join(BASE_DIR, "make_tables.py")

for p in (FIGS_PY, TABS_PY):
    print(("OK   " if os.path.exists(p) else "MISS "), p)

assert os.path.exists(FIGS_PY), "upload make_figures.py to " + BASE_DIR

In [ ]:
FIGS = os.path.join(RDIR, "figures")
os.makedirs(FIGS, exist_ok=True)

r = subprocess.run([sys.executable, FIGS_PY,
                    "--report_dir", RDIR,
                    "--out_dir", FIGS],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr.strip():
    print("--- stderr ---")
    print(r.stderr[-2500:])
print("return code:", r.returncode)

In [ ]:
from IPython.display import Image, display
display(Image(filename=os.path.join(FIGS, "fig2_offbyone_step.png")))

In [ ]:
display(Image(filename=os.path.join(FIGS, "fig1_error_budget.png")))

In [ ]:
display(Image(filename=os.path.join(FIGS, "fig3_selfval.png")))

In [ ]:
if os.path.exists(TABS_PY):
    TABS = os.path.join(RDIR, "paper_tables")
    os.makedirs(TABS, exist_ok=True)
    r = subprocess.run([sys.executable, TABS_PY,
                        "--report_dir", RDIR,
                        "--out_dir", TABS],
                       capture_output=True, text=True)
    print(r.stdout)
    if r.stderr.strip():
        print(r.stderr[-2000:])
else:
    print("make_tables.py not uploaded - skipping")

In [ ]:
from IPython.display import Markdown

p = os.path.join(RDIR, "paper_tables", "tables.md")
if os.path.exists(p):
    display(Markdown(open(p).read()))
else:
    print("tables.md not found - run Cell 7")

In [ ]:
import shutil

BUNDLE = os.path.join(BASE_DIR, "results", "paper_assets")
shutil.rmtree(BUNDLE, ignore_errors=True)
os.makedirs(BUNDLE, exist_ok=True)

# figures
for f in glob.glob(os.path.join(FIGS, "*")):
    shutil.copy2(f, BUNDLE)

# tables
tdir = os.path.join(RDIR, "paper_tables")
if os.path.isdir(tdir):
    for f in glob.glob(os.path.join(tdir, "*")):
        shutil.copy2(f, BUNDLE)

# raw report CSVs for the appendix / artifact release
for f in glob.glob(os.path.join(RDIR, "*.csv")) + \
         glob.glob(os.path.join(RDIR, "summary.json")):
    shutil.copy2(f, BUNDLE)

zip_path = shutil.make_archive(BUNDLE, "zip", BUNDLE)
print("bundle:", zip_path)
print("%.1f KB" % (os.path.getsize(zip_path) / 1024))
print()
for f in sorted(os.listdir(BUNDLE)):
    print("  ", f)
print("\nDownload from the Notebooks file browser:")
print("  results/paper_assets.zip")

In [ ]:
import os, sys, glob, json, subprocess
import pandas as pd

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

BASE_DIR = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/verifier-error-budget"
SRC = os.path.join(BASE_DIR, "src")
if not os.path.isdir(SRC):
    SRC = BASE_DIR
os.chdir(BASE_DIR)
if SRC not in sys.path:
    sys.path.insert(0, SRC)

ml = MLClient(DefaultAzureCredential(),
              "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
              "AIModels", "Reinforcementinfra")

print("BASE_DIR:", BASE_DIR)
print("SRC     :", SRC)

In [ ]:
checks = {
    os.path.join(SRC, "aggregate.py"): "n_eval",
    os.path.join(BASE_DIR, "make_tables.py"): "Self-val. (judged)",
    os.path.join(BASE_DIR, "make_figures.py"): "v2 FIX",
}
ok = True
for path, marker in checks.items():
    if not os.path.exists(path):
        print("MISSING ", path); ok = False; continue
    hit = marker in open(path).read()
    print(("v6 OK   " if hit else "OLD FILE"), os.path.basename(path))
    ok &= hit

print()
assert ok, "replace the three v6 files before continuing"
print("all three files are the new versions")

In [ ]:
cands = sorted(glob.glob(os.path.join(BASE_DIR, "results", "*_verdicts")),
               key=os.path.getmtime, reverse=True)
print("verdict dirs (newest first):")
for c in cands:
    n = len(glob.glob(os.path.join(c, "**", "*.parquet"), recursive=True))
    print(f"    {n:5d} shards   {c}")

assert cands, "no *_verdicts folder - download verdicts first"

pq = glob.glob(os.path.join(cands[0], "**", "*.parquet"), recursive=True)
assert pq, "no parquet inside " + cands[0]
VDIR = os.path.dirname(pq[0])
print("\nusing VDIR =", VDIR)
print("shards     =", len(pq))

In [ ]:
RDIR6 = os.path.join(BASE_DIR, "results", "report_v6")
os.makedirs(RDIR6, exist_ok=True)

r = subprocess.run([sys.executable, os.path.join(SRC, "aggregate.py"),
                    "--verdicts_dir", VDIR,
                    "--report_dir", RDIR6],
                   capture_output=True, text=True)

print(r.stdout[-6000:])
if r.stderr.strip():
    print("--- stderr ---")
    print(r.stderr[-2500:])
print("return code:", r.returncode)
assert r.returncode == 0

In [ ]:
t0 = pd.read_csv(os.path.join(RDIR6, "T0_raw_counts.csv"))
low = t0[(t0.coverage < 0.999) & (t0.n > 0)].sort_values("coverage")

print("=" * 72)
print("LOW COVERAGE CELLS")
print("=" * 72)
if len(low):
    cols = ["tclass", "stratum", "verifier", "n", "n_eval",
            "coverage", "err_rate"]
    print(low[cols].round(4).to_string(index=False))
else:
    print("none - every verifier returned a verdict on every input")
print("=" * 72)

In [ ]:
TABS = os.path.join(RDIR6, "paper_tables")
FIGS = os.path.join(RDIR6, "figures")

for script, out in [("make_tables.py", TABS), ("make_figures.py", FIGS)]:
    os.makedirs(out, exist_ok=True)
    r = subprocess.run([sys.executable, os.path.join(BASE_DIR, script),
                        "--report_dir", RDIR6, "--out_dir", out],
                       capture_output=True, text=True)
    print("---", script, "---")
    print(r.stdout)
    if r.stderr.strip():
        print(r.stderr[-1500:])

In [ ]:
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 300)

L = []
def P(*a):
    s = " ".join(str(x) for x in a); L.append(s); print(s)

P("=" * 74)
P("VERIFIER ERROR BUDGET - v6 DIGEST (coverage-aware)")
P("=" * 74)

P("\n### SUMMARY")
P(json.dumps(json.load(open(os.path.join(RDIR6, "summary.json"))), indent=2))

def show(fn, title, cols=None):
    p = os.path.join(RDIR6, fn)
    if not os.path.exists(p):
        P("\n### " + title + " (missing)"); return None
    df = pd.read_csv(p)
    if cols:
        cols = [c for c in cols if c in df.columns]
        df = df[cols]
    P("\n### " + title)
    P(df.round(4).to_string(index=False))
    return df

show("T6_reconciliation.csv", "T6 RECONCILIATION",
     ["verifier","n","n_eval","coverage","n_true","n_false","n_error",
      "self_validation_all","self_validation_rate","residual"])

show("T1_fn_by_stratum_verifier.csv", "T1 IN-CONTRACT FN",
     ["stratum","verifier","n","n_eval","coverage","fn_rate","fn_rate_all",
      "err_rate","fn_ci_lo","fn_ci_hi"])

show("T2_error_mass_share.csv", "T2 ERROR MASS SHARE",
     ["verifier","stratum","fail_count","error_mass_share",
      "share_from_reject","share_from_error"])

show("T3_fp_by_stratum_verifier.csv", "T3 ADVERSARIAL FP",
     ["stratum","verifier","n","n_eval","coverage","fp_rate","fp_rate_all",
      "err_rate","ci_lo","ci_hi"])

show("T8_offbyone_by_magnitude.csv", "T8 OFF-BY-ONE BY MAGNITUDE")
show("T7_out_of_contract.csv", "T7 OUT-OF-CONTRACT")
show("T5_contract_dependent.csv", "T5 CONTRACT-DEPENDENT")

P("\n### T4 CROSS-VERIFIER DISAGREEMENT")
P(pd.read_csv(os.path.join(RDIR6, "T4_disagreement_matrix.csv"),
              index_col=0).round(4).to_string())

p = os.path.join(RDIR6, "T4b_disagreement_support.csv")
if os.path.exists(p):
    P("\n### T4b PAIR SUPPORT (both verifiers returned a verdict)")
    P(pd.read_csv(p, index_col=0).to_string())

P("\n" + "=" * 74)
open(os.path.join(RDIR6, "digest_v6.txt"), "w").write("\n".join(L))
print("\n[saved digest_v6.txt]")

In [ ]:
from IPython.display import Image, display
for f in ["fig3_selfval", "fig2_offbyone_step", "fig1_error_budget"]:
    p = os.path.join(FIGS, f + ".png")
    if os.path.exists(p):
        print(f)
        display(Image(filename=p))

In [ ]:
import shutil

BUNDLE = os.path.join(BASE_DIR, "results", "paper_assets_v6")
shutil.rmtree(BUNDLE, ignore_errors=True)
os.makedirs(BUNDLE, exist_ok=True)

for src in [FIGS, TABS]:
    for f in glob.glob(os.path.join(src, "*")):
        shutil.copy2(f, BUNDLE)
for f in glob.glob(os.path.join(RDIR6, "*.csv")) + \
         glob.glob(os.path.join(RDIR6, "*.json")) + \
         glob.glob(os.path.join(RDIR6, "*.txt")):
    shutil.copy2(f, BUNDLE)

zip_path = shutil.make_archive(BUNDLE, "zip", BUNDLE)
print("bundle:", zip_path)
print("%.1f KB" % (os.path.getsize(zip_path) / 1024))
print()
for f in sorted(os.listdir(BUNDLE)):
    print("  ", f)
print("\nDownload from Notebooks file browser:")
print("  results/paper_assets_v6.zip")